# TP3 - Qwen

En Colab: activar GPU T4 y ejecutar las celdas en orden. Si la carpeta tiene otro nombre, cambiar `RAIZ`. La evaluación de test está desactivada porque ya se hizo.

In [1]:
%pip -q install "transformers>=4.45,<5" accelerate scikit-learn pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os
import re
import time
import pandas as pd
import torch
from google.colab import drive
from sklearn.metrics import f1_score, classification_report

drive.mount("/content/drive")
RAIZ = "/content/drive/MyDrive/TP IA y Neuro"
DATOS = f"{RAIZ}/data"
RES = f"{RAIZ}/resultados"
os.makedirs(RES, exist_ok=True)

clases = ["favor", "against", "neutral"]

def preparar(df):
    df = df.copy()
    df["majority_label"] = df["majority_label"].astype("string").str.strip().str.lower()
    df = df[df["majority_label"].isin(clases)].copy()
    df["text"] = df["text"].fillna("").astype(str)
    return df.reset_index(drop=True)

train = preparar(pd.read_csv(f"{DATOS}/train.csv"))
val = preparar(pd.read_csv(f"{DATOS}/val.csv"))
print("Train:", len(train), "| Val:", len(val), "| GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activar GPU T4 en Colab"

Mounted at /content/drive
Train: 1995 | Val: 428 | GPU: True


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

nombre = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(nombre)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    nombre, dtype=torch.float16, device_map="auto"
)
model.eval()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [4]:
# Un ejemplo de cada clase del conjunto de entrenamiento
ejemplos = train.groupby("majority_label", group_keys=False).sample(n=1, random_state=42)
texto_ejemplos = "\n\n".join(
    f"{fila['text']}\nEtiqueta: {fila['majority_label']}"
    for _, fila in ejemplos.iterrows()
)

def crear_prompt(texto, modo):
    if modo == "zero-shot-mejorado":
        instrucciones = """
Sos un especialista en análisis de posturas (stance detection).

Tu tarea es determinar la postura del comentario
respecto de la propuesta presentada.

Clasificá usando exactamente una de estas etiquetas:

- favor: el comentario apoya, defiende o está de acuerdo
  con la propuesta.

- against: el comentario rechaza, critica o expresa
  desacuerdo con la propuesta.

- neutral: el comentario no expresa una postura clara,
  es ambiguo o no está relacionado con la propuesta.

REGLAS IMPORTANTES:
1. Evaluá la postura respecto de la PROPUESTA, no el
   sentimiento general del comentario.
2. Prestá atención a las negaciones y al contexto.
3. No confundas una crítica a los opositores de la
   propuesta con una crítica a la propuesta misma.
4. Utilizá neutral cuando no exista evidencia suficiente
   para determinar una postura.
5. Respondé únicamente con: favor, against o neutral.
"""
    elif modo in ["zero-shot", "few-shot"]:
        instrucciones = """
Clasificá la postura del comentario respecto de la propuesta.

Las únicas etiquetas posibles son:
- favor: apoya la propuesta.
- against: se opone a la propuesta.
- neutral: no expresa una postura clara.

Respondé únicamente con una de estas tres palabras:
favor, against, neutral.
"""
        if modo == "few-shot":
            instrucciones += "\nEjemplos de clasificación:\n" + texto_ejemplos + "\n\n"
    else:
        raise ValueError("Modo no válido")

    instrucciones += f"\nTexto a clasificar:\n{texto}\n\nEtiqueta:"
    mensajes = [{"role": "user", "content": instrucciones}]
    return tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )

In [5]:
def evaluar_llm(datos, modo, batch_size=4):
    prompts = [crear_prompt(texto, modo) for texto in datos["text"]]
    respuestas = []
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inicio = time.perf_counter()

    for i in range(0, len(prompts), batch_size):
        lote = prompts[i:i + batch_size]
        inputs = tokenizer(
            lote, return_tensors="pt", padding=True,
            truncation=True, max_length=1536,
            add_special_tokens=False
        ).to(model.device)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, max_new_tokens=10, do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        generados = outputs[:, inputs["input_ids"].shape[1]:]
        respuestas.extend(tokenizer.batch_decode(generados, skip_special_tokens=True))

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    tiempo = time.perf_counter() - inicio

    predicciones = []
    invalidas = 0
    for respuesta in respuestas:
        match = re.search(r"\b(favor|against|neutral)\b", respuesta.lower())
        if match:
            predicciones.append(match.group(1))
        else:
            invalidas += 1
            predicciones.append("neutral")  # Criterio usado en los experimentos originales

    f1 = f1_score(datos["majority_label"], predicciones,
                  labels=clases, average="macro", zero_division=0)
    print(f"{modo}: F1={f1:.4f}, tiempo={tiempo:.2f}s, inválidas={invalidas}")
    return {"macro_f1": f1, "tiempo": tiempo,
            "invalidas": invalidas, "predicciones": predicciones,
            "respuestas": respuestas}

In [6]:
# Validación: comparar los tres prompts
EJECUTAR_VALIDACION = True

if EJECUTAR_VALIDACION:
    resultados = []
    predicciones_val = val[["majority_label"]].copy()
    for modo in ["zero-shot", "few-shot", "zero-shot-mejorado"]:
        r = evaluar_llm(val, modo)
        resultados.append({
            "Modelo": "Qwen2.5-1.5B " + modo,
            "Macro-F1": r["macro_f1"],
            "Entrenamiento (s)": 0,
            "Inferencia (s)": r["tiempo"],
            "Respuestas inválidas": r["invalidas"]
        })
        predicciones_val[modo] = r["predicciones"]
    tabla_val = pd.DataFrame(resultados)
    display(tabla_val)
    tabla_val.to_csv(f"{RES}/resultados_llm.csv", index=False)
    predicciones_val.to_csv(f"{RES}/predicciones_llm.csv", index=False)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


zero-shot: F1=0.3964, tiempo=29.32s, inválidas=0
few-shot: F1=0.4083, tiempo=47.44s, inválidas=0
zero-shot-mejorado: F1=0.4472, tiempo=36.83s, inválidas=0


,Modelo,Macro-F1,Entrenamiento (s),Inferencia (s),Respuestas inválidas
0,Qwen2.5-1.5B zero-shot,0.396416,0,29.322696,0
1,Qwen2.5-1.5B few-shot,0.408291,0,47.436305,0
2,Qwen2.5-1.5B zero-shot-mejorado,0.447216,0,36.827224,0


In [7]:
# Test: se deja desactivado porque ya fue evaluado con el mejor prompt
# Para reproducir la evaluación final, cambiar a True.
EJECUTAR_TEST = False

if EJECUTAR_TEST:
    test = preparar(pd.read_csv(f"{DATOS}/test.csv"))
    r = evaluar_llm(test, "zero-shot-mejorado")
    print(classification_report(test["majority_label"], r["predicciones"],
                                labels=clases, zero_division=0))
    pred_test = test[["majority_label"]].copy()
    if "id" in test.columns:
        pred_test.insert(0, "id", test["id"])
    pred_test["prediccion"] = r["predicciones"]
    pred_test["respuesta_cruda"] = r["respuestas"]
    pred_test.to_csv(f"{RES}/test_llm.csv", index=False)
    pd.DataFrame([{
        "Modelo": "Qwen2.5-1.5B Prompt mejorado",
        "Macro-F1 Test": r["macro_f1"],
        "Inferencia Test (s)": r["tiempo"],
        "Respuestas inválidas": r["invalidas"],
        "N Test": len(test)
    }]).to_csv(f"{RES}/resumen_test_llm.csv", index=False)